In [1]:
!pip -q install -U ultralytics rasterio pyyaml tifffile geopandas shapely pyogrio
# ultralytics for yolo training, pyyaml for YAML files

In [2]:
from pathlib import Path

import random # for train/validation/test split
import shutil 

import xml.etree.ElementTree as ET # read annotation files
import yaml # writes YOLO YAML

import rasterio
import geopandas as gpd
from shapely.geometry import Polygon


base = Path("C:/yolo_scoping/50cm")
export_dir = base / "chips_train" / "2025" / "export_arc_with_nofeature_tiles_320sizestride_06"

raw_ds = base / "yolo_datasets" / "2025_trainset"
rgb_ds = base / "yolo_datasets" / "2025_trainset_rgb3"

runs_dir = base / "runs"

print("export_dir:", export_dir)
print("raw_ds:", raw_ds)
print("rgb_ds:", rgb_ds)

export_dir: C:\yolo_scoping\50cm\chips_train\2025\export_arc_with_nofeature_tiles_320sizestride_06
raw_ds: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset
rgb_ds: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3


In [4]:

# Build YOLO dataset from ArcGIS VOC export

seed = 42

train_frac = 0.6
val_frac = 0.3
test_frac = 0.1

# Trees as class 0
classes = ["0"]

# Drop tile-edge slivers
min_px = 3

# Keep chips with no trees
keep_empty_tiles = True

img_exts = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}

# input chips and lables, output YOLO dataset
images_dir = export_dir / "images"
xml_dir = export_dir / "labels"
out_dir = raw_ds

# read one xml
def parse_voc(xml_path: Path):
    """Read one ArcGIS VOC XML file."""
    # load the xml structure
    root = ET.parse(xml_path).getroot()
    # read image size
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)
    # for storing bounding boxes from xml
    boxes = []
 

    
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()

        if name not in classes:
            continue

        class_id = classes.index(name)

        b = obj.find("bndbox")
        xmin = float(b.find("xmin").text)
        ymin = float(b.find("ymin").text)
        xmax = float(b.find("xmax").text)
        ymax = float(b.find("ymax").text)

        boxes.append((class_id, xmin, ymin, xmax, ymax))

    return w, h, boxes




def voc_to_yolo_line(class_id, xmin, ymin, xmax, ymax, w, h):
    """Convert one VOC box into one YOLO line."""

    xmin = max(0.0, min(xmin, w))
    xmax = max(0.0, min(xmax, w))
    ymin = max(0.0, min(ymin, h))
    ymax = max(0.0, min(ymax, h))

    bw = xmax - xmin
    bh = ymax - ymin

    # skip if boxes smaller than three pixels
    if bw < min_px or bh < min_px:
        return None
    # find x y  center and normalise, YOLO wants normalised position
    x_c = (xmin + xmax) / 2.0 / w
    y_c = (ymin + ymax) / 2.0 / h
    # build yolo label row
    # center x, center y, width, height
    return f"{class_id} {x_c:.6f} {y_c:.6f} {bw / w:.6f} {bh / h:.6f}"


def make_yolo_dirs(ds: Path):
    for split in ["train", "val", "test"]:
        (ds / "images" / split).mkdir(parents=True, exist_ok=True)
        (ds / "labels" / split).mkdir(parents=True, exist_ok=True)


def build_yolo_dataset():
    random.seed(seed)
    make_yolo_dirs(out_dir)

    # Find chips that have matching VOC labels
    items = []

    for img in images_dir.iterdir():
        if img.suffix.lower() not in img_exts:
            continue

 
        # Keep every image chip.if it doesnt have xml it is treated as no-tree background 
        items.append(img.name)
    if not items:
        raise RuntimeError(f"No image chips found in {images_dir}")

    # Random split, a spatial split may be needed later
    random.shuffle(items)

    n = len(items)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    splits = {
        "train": items[:n_train],
        "val": items[n_train:n_train + n_val],
        "test": items[n_train + n_val:],
    }

    stats = {
        "chips_total": n,
        "images_written": 0,
        "labels_written": 0,
        "voc_boxes_total": 0,
        "yolo_boxes_kept": 0,
        "yolo_boxes_skipped_tiny": 0,
        "background_tiles": 0,
    }

    for split, files in splits.items():
        for fn in files:
            img_src = images_dir / fn
            xml_src = xml_dir / f"{Path(fn).stem}.xml"


            if xml_src.exists():
                w, h, boxes = parse_voc(xml_src)
            else:
                # No XML means no annotated objects, so treat as background tile
                with rasterio.open(img_src) as src:
                    w = src.width
                    h = src.height
                boxes = []

            stats["voc_boxes_total"] += len(boxes)

            lines = []
            # Convert each VOC box to YOLO format
            for class_id, xmin, ymin, xmax, ymax in boxes:
                line = voc_to_yolo_line(class_id, xmin, ymin, xmax, ymax, w, h)

                if line is None:
                    stats["yolo_boxes_skipped_tiny"] += 1
                    continue

                lines.append(line)
                stats["yolo_boxes_kept"] += 1

            # count as a background tile if no valid box
            if len(lines) == 0:
                stats["background_tiles"] += 1

                if split in {"train", "val"} and not keep_empty_tiles:
                    continue

            shutil.copy2(img_src, out_dir / "images" / split / fn)

            # yolo label text
            label_out = out_dir / "labels" / split / f"{Path(fn).stem}.txt"
            label_out.write_text("\n".join(lines), encoding="utf-8")

            stats["images_written"] += 1
            stats["labels_written"] += 1

    data = {
        "path": str(out_dir),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "tree"},
    }

    # write data.yaml for yolo training 
    with open(out_dir / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)

    print("Built YOLO dataset:", out_dir)
    print("Split sizes:", {k: len(v) for k, v in splits.items()})
    print("Stats:", stats)
    print("data.yaml:", out_dir / "data.yaml")


build_yolo_dataset()

Built YOLO dataset: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset
Split sizes: {'train': 15, 'val': 7, 'test': 4}
Stats: {'chips_total': 26, 'images_written': 26, 'labels_written': 26, 'voc_boxes_total': 56, 'yolo_boxes_kept': 56, 'yolo_boxes_skipped_tiny': 0, 'background_tiles': 8}
data.yaml: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset\data.yaml


In [7]:
# Convert training chips to 3-band images

def make_rgb_tiles(src_dir: Path, dst_dir: Path):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    n_in = 0
    n_out = 0

    for tif in src_dir.glob("*.tif"):
        n_in += 1

        with rasterio.open(tif) as src:
            if src.count < 3:
                continue

            # Keep the first three bands for YOLO
            rgb = src.read([1, 2, 3])

            profile = src.profile.copy()
            profile.update(count=3)

        with rasterio.open(dst_dir / tif.name, "w", **profile) as dst:
            dst.write(rgb)

        n_out += 1

    print(f"Input tiles: {n_in}, wrote 3-band rgb tiles: {n_out}")
    print("Output:", dst_dir)

# Convert train, val, and test images to 3-band RGB 
for split in ["train", "val", "test"]:
    make_rgb_tiles(
        raw_ds / "images" / split,
        rgb_ds / "images" / split,
    )

# Copy labels from raw dataset to RGB dataset
for split in ["train", "val", "test"]:
    src = raw_ds / "labels" / split
    dst = rgb_ds / "labels" / split
    dst.mkdir(parents=True, exist_ok=True)

    for p in src.glob("*.txt"):
        shutil.copy2(p, dst / p.name)

data = {
    "path": str(rgb_ds).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "tree"},
}

with open(rgb_ds / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("RGB3 dataset:", rgb_ds / "data.yaml")

Input tiles: 20, wrote 3-band rgb tiles: 20
Output: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\train
Input tiles: 11, wrote 3-band rgb tiles: 11
Output: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\val
Input tiles: 7, wrote 3-band rgb tiles: 7
Output: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test
RGB3 dataset: C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\data.yaml


In [8]:
# Train YOLO
!yolo detect train \
  data="C:/yolo_scoping/50cm/yolo_datasets/2025_trainset_rgb3/data.yaml" \
  model="yolo11n.pt" \
  imgsz=320 epochs=100 batch=8 patience=30 \
  project="C:/yolo_scoping/50cm/runs" \
  name="train_2025_rgb3_y11n" \
  exist_ok=True

Ultralytics 8.4.45  Python-3.10.20 torch-2.11.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/yolo_scoping/50cm/yolo_datasets/2025_trainset_rgb3/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_2025_rgb3_y11n, nbs=64, nms=False, opset=None, optimize=Fals

[ WARN:0@7.791] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@7.793] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@7.793] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@7.793] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@7.817] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@7.817] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@7.817] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@7.817] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 347

In [10]:
# Predict on held-out 2025 test chips

!yolo detect predict \
  model="C:/yolo_scoping/50cm/runs/train_2025_rgb3_y11n/weights/best.pt" \
  source="C:/yolo_scoping/50cm/yolo_datasets/2025_trainset_rgb3/images/test" \
  imgsz=320 conf=0.5 iou=0.5 \
  save=True save_txt=True save_conf=True \
  project="C:/yolo_scoping/50cm/runs" \
  name="predict_2025_test" \
  exist_ok=True

Ultralytics 8.4.45  Python-3.10.20 torch-2.11.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

image 1/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000000.tif: 320x320 1 tree, 49.9ms
image 2/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000003.tif: 320x320 1 tree, 30.0ms
image 3/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000011.tif: 320x320 2 trees, 33.2ms
image 4/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000013.tif: 320x320 2 trees, 31.4ms
image 5/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000019.tif: 320x320 1 tree, 31.3ms
image 6/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000038.tif: 320x320 2 trees, 32.9ms
image 7/7 C:\yolo_scoping\50cm\yolo_datasets\2025_trainset_rgb3\images\test\000000000046.tif: 320x320 (no detections), 39.4m

[ WARN:0@3.424] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@3.425] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@3.425] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@3.425] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@3.614] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@3.614] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@3.614] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@3.614] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 347

In [11]:
# Count images and labels in each split

for split in ["train", "val", "test"]:
    imgs = list((rgb_ds / "images" / split).glob("*.tif"))
    lbls = list((rgb_ds / "labels" / split).glob("*.txt"))

    n_boxes = 0

    for txt in lbls:
        content = txt.read_text().strip()

        if content:
            n_boxes += len(content.splitlines())

    print(
        split,
        "images:", len(imgs),
        "label_files:", len(lbls),
        "total_boxes:", n_boxes,
    )

train images: 20 label_files: 20 total_boxes: 50
val images: 11 label_files: 11 total_boxes: 15
test images: 7 label_files: 7 total_boxes: 19


In [12]:
#  2012 and 2019 chips to rgb files

make_rgb_tiles(
    base / "chips_predict" / "2012" / "images",
    base / "chips_predict" / "2012" / "images_rgb3",
)

make_rgb_tiles(
    base / "chips_predict" / "2019" / "images",
    base / "chips_predict" / "2019" / "images_rgb3",
)

Input tiles: 36, wrote 3-band rgb tiles: 36
Output: C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3
Input tiles: 26, wrote 3-band rgb tiles: 26
Output: C:\yolo_scoping\50cm\chips_predict\2019\images_rgb3


In [13]:
# Predict trees in 2012 and 2019 chips


!yolo detect predict \
  model="C:/yolo_scoping/50cm/runs/train_2025_rgb3_y11n/weights/best.pt" \
  source="C:/yolo_scoping/50cm/chips_predict/2012/images_rgb3" \
  imgsz=320 conf=0.5 iou=0.5 \
  save=True save_txt=True save_conf=True \
  project="C:/yolo_scoping/50cm/runs" \
  name="predict_2012" \
  exist_ok=True



Ultralytics 8.4.45  Python-3.10.20 torch-2.11.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

image 1/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000000.tif: 320x320 (no detections), 66.0ms
image 2/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000001.tif: 320x320 (no detections), 33.2ms
image 3/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000002.tif: 320x320 (no detections), 34.6ms
image 4/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000003.tif: 320x320 (no detections), 39.5ms
image 5/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000004.tif: 320x320 1 tree, 34.1ms
image 6/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000005.tif: 320x320 (no detections), 30.7ms
image 7/36 C:\yolo_scoping\50cm\chips_predict\2012\images_rgb3\000000000006.tif: 320x320 (no detections), 32.0ms
image 8/36 C:\yolo_scoping\50cm\chips_predict\2

[ WARN:0@3.586] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@3.586] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@3.586] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@3.586] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34736 (0x87b0) encountered
[ WARN:0@3.586] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@3.776] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@3.776] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@3.776] global grfmt_tiff.cpp:122 cv::TIFF_Warning TIFFReadDirectory: Unknown field with tag 347

In [ ]:
!yolo detect predict \
  model="C:/yolo_scoping/50cm/runs/train_2025_rgb3_y11n/weights/best.pt" \
  source="C:/yolo_scoping/50cm/chips_predict/2019/images_rgb3" \
  imgsz=320 conf=0.5 iou=0.5 \
  save=True save_txt=True save_conf=True \
  project="C:/yolo_scoping/50cm/runs" \
  name="predict_2019" \
  exist_ok=True

In [ ]:
# yolo text and tif to spatial gpkg
def yolo_txt_to_gpkg(image_dir: Path, label_dir: Path, out_gpkg: Path, year: int, layer="tree_boxes"):
    features = []
    crs = None

    for tif_path in image_dir.glob("*.tif"):
        txt_path = label_dir / f"{tif_path.stem}.txt"

        
        # skip if no txt prediction exists
        if not txt_path.exists():
            continue

        txt = txt_path.read_text(encoding="utf-8").strip()

        # skip if exists but empty
        if not txt:
            continue

        with rasterio.open(tif_path) as src:
            width = src.width
            height = src.height
            crs = src.crs

            for line in txt.splitlines():
                parts = line.split()

                if len(parts) < 5:
                    continue

                class_id = int(float(parts[0]))
                
                # read normalized YOLO coordinates
                x_center, y_center, box_w, box_h = map(float, parts[1:5])
                conf = float(parts[5]) if len(parts) > 5 else None

                x_c = x_center * width
                y_c = y_center * height
                bw = box_w * width
                bh = box_h * height

                x_min = int(max(0, min(round(x_c - bw / 2), width)))
                x_max = int(max(0, min(round(x_c + bw / 2), width)))
                y_min = int(max(0, min(round(y_c - bh / 2), height)))
                y_max = int(max(0, min(round(y_c + bh / 2), height)))

                ul = src.transform * (x_min, y_min)
                ur = src.transform * (x_max, y_min)
                lr = src.transform * (x_max, y_max)
                ll = src.transform * (x_min, y_max)

                polygon = Polygon([ul, ur, lr, ll, ul])

                features.append({
                    "year": year,
                    "image": tif_path.name,
                    "class_id": class_id,
                    "class_name": "tree",
                    "confidence": conf,
                    "geometry": polygon,
                })

    if not features:
        raise ValueError(f"No detections found in {label_dir}")

    gdf = gpd.GeoDataFrame(features, geometry="geometry", crs=crs)
    out_gpkg.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(out_gpkg, layer=layer, driver="GPKG")

    print(f"Saved {len(gdf)} boxes to {out_gpkg}")

In [ ]:
yolo_txt_to_gpkg(
    image_dir=rgb_ds / "images" / "test",
    label_dir=runs_dir / "predict_2025_test" / "labels",
    out_gpkg=runs_dir / "predict_2025_test" / "tree_boxes_2025.gpkg",
    year=2025,
)

yolo_txt_to_gpkg(
    image_dir=base / "chips_predict" / "2012" / "images_rgb3",
    label_dir=runs_dir / "predict_2012" / "labels",
    out_gpkg=runs_dir / "predict_2012" / "tree_boxes_2012.gpkg",
    year=2012,
)

yolo_txt_to_gpkg(
    image_dir=base / "chips_predict" / "2019" / "images_rgb3",
    label_dir=runs_dir / "predict_2019" / "labels",
    out_gpkg=runs_dir / "predict_2019" / "tree_boxes_2019.gpkg",
    year=2019,
)